# Track AI Agent Costs

AI agents can burn through tokens fast. A single Claude Code session with subagents can use 100k+ tokens across dozens of API calls. This notebook shows how to track what those calls actually cost, from a single response to an entire project.

We'll cover:
1. Estimating cost from any API response (zero setup)
2. Tracking costs across a session from JSONL transcripts
3. Analyzing cache savings
4. Setting up real-time cost visibility via MCP tools

## Setup

Install the LLMKit SDK, which bundles pricing data for 730+ models across 11 providers.

In [1]:
%pip install llmkit-sdk anthropic python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


C:\f3d1\claude-cookbooks\.venv\Scripts\python.exe: No module named pip


## 1. Estimate cost from a single response

The simplest approach: wrap your HTTP client with `tracked()`. It intercepts responses and estimates costs from the `usage` field using a bundled pricing table. No API key, no server, no config.

In [2]:
from dotenv import load_dotenv

load_dotenv()

import anthropic
from llmkit import tracked

# tracked() returns an httpx.Client that estimates costs automatically
client = anthropic.Anthropic(http_client=tracked())

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)

print(response.content[0].text)
print(f"\nTokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out")

The capital of France is Paris.

Tokens: 14 in, 10 out


You can also estimate the cost of any response after the fact with `estimate_cost()`:

In [3]:
from llmkit import calculate_cost

# calculate_cost works with any model in the pricing table
# use it standalone with token counts from any source
cost = calculate_cost(
    model="claude-haiku-4-5",
    input_tokens=350,
    output_tokens=120,
)

if cost is not None:
    print(f"Estimated cost: ${cost:.6f}")
else:
    print("Model not in pricing table")

# works for any model across 11 providers
for model in ["claude-sonnet-4-6", "claude-opus-4-6", "gpt-4o", "gemini-2.0-flash"]:
    c = calculate_cost(model, input_tokens=1000, output_tokens=500)
    if c is not None:
        print(f"  {model}: ${c:.4f} for 1k in + 500 out")

Estimated cost: $0.000950
  claude-sonnet-4-6: $0.0105 for 1k in + 500 out
  claude-opus-4-6: $0.0175 for 1k in + 500 out
  gpt-4o: $0.0075 for 1k in + 500 out
  gemini-2.0-flash: $0.0003 for 1k in + 500 out


## 2. Track session costs from transcripts

Claude Code stores conversation transcripts as JSONL files in `~/.claude/projects/`. Each assistant message includes token usage. Let's parse a session transcript and calculate the total cost.

We'll use a sample transcript included with this notebook (the same format Claude Code writes).

In [4]:
import json
from pathlib import Path

from llmkit import calculate_cost

# parse a Claude Code session transcript
transcript = Path("observability/track_agent_costs/sample_session.jsonl")
if not transcript.exists():
    transcript = Path("sample_session.jsonl")  # fallback for local runs

total_cost = 0.0
total_input = 0
total_output = 0
total_cache_read = 0
total_cache_write = 0
messages = 0

for line in transcript.read_text().strip().split("\n"):
    data = json.loads(line)
    if data.get("type") != "assistant":
        continue

    msg = data["message"]
    usage = msg.get("usage", {})
    model = msg.get("model", "")

    inp = usage.get("input_tokens", 0)
    out = usage.get("output_tokens", 0)
    cache_read = usage.get("cache_read_input_tokens", 0)
    cache_write = usage.get("cache_creation_input_tokens", 0)

    cost = calculate_cost(model, inp, out, cache_read, cache_write)
    if cost is not None:
        total_cost += cost

    total_input += inp
    total_output += out
    total_cache_read += cache_read
    total_cache_write += cache_write
    messages += 1

print(f"Session: {messages} messages")
print(f"Tokens: {total_input:,} input, {total_output:,} output")
print(f"Cache: {total_cache_read:,} read, {total_cache_write:,} write")
print(f"Estimated cost: ${total_cost:.4f}")

Session: 5 messages
Tokens: 5,460 input, 6,530 output
Cache: 49,400 read, 4,750 write
Estimated cost: $0.1470


## 3. Analyze cache savings

Prompt caching can save significant money. Cache reads cost ~10% of regular input tokens. Let's see how much caching saved in this session.

In [5]:
from llmkit import calculate_cost

# compare what cache reads cost vs what they would cost at full price
# claude-sonnet-4-6: $3/M input, $0.30/M cache read (90% savings)
INPUT_PRICE_PER_M = 3.0
CACHE_READ_PRICE_PER_M = 0.30

# what cache reads WOULD have cost at full input price
full_price = (total_cache_read / 1_000_000) * INPUT_PRICE_PER_M
# what they actually cost at cache read price
cache_price = (total_cache_read / 1_000_000) * CACHE_READ_PRICE_PER_M
savings = full_price - cache_price

print(f"Cache read tokens: {total_cache_read:,}")
print(f"Full price would be: ${full_price:.4f}")
print(f"Cache price was: ${cache_price:.4f}")
print(f"Savings: ${savings:.4f} ({savings / full_price * 100:.0f}% reduction)")

if total_cache_write > 0:
    ratio = total_cache_read / total_cache_write
    print(f"\nCache efficiency: {ratio:.1f}x read-to-write ratio")
    print("(Higher is better. Above 3x means caching is paying for itself)")

Cache read tokens: 49,400
Full price would be: $0.1482
Cache price was: $0.0148
Savings: $0.1334 (90% reduction)

Cache efficiency: 10.4x read-to-write ratio
(Higher is better. Above 3x means caching is paying for itself)


## 4. Real-time cost visibility with MCP

For continuous cost tracking inside Claude Code, Cursor, or Cline, the [LLMKit MCP server](https://github.com/smigolsmigol/llmkit/tree/main/packages/mcp-server) provides 11 tools that surface cost data directly in your editor.

**Local tools (no API key needed):**
- `llmkit_local_session` - current session cost
- `llmkit_local_projects` - cumulative cost across all projects
- `llmkit_local_cache` - cache savings analysis
- `llmkit_local_forecast` - monthly spend projection
- `llmkit_local_agents` - subagent cost attribution

### Setup

Add to your `.mcp.json` (Claude Code) or `.cursor/mcp.json` (Cursor):

```json
{
  "mcpServers": {
    "llmkit": {
      "command": "npx",
      "args": ["@f3d1/llmkit-mcp-server"]
    }
  }
}
```

No API key needed for local tools. They read Claude Code's JSONL transcripts directly.

### Auto-log with SessionEnd hook

Add to Claude Code's `settings.json` to automatically log costs when a session ends:

```json
{
  "hooks": {
    "SessionEnd": [
      {
        "type": "command",
        "command": "npx @f3d1/llmkit-mcp-server --hook"
      }
    ]
  }
}
```

This parses the session transcript and prints a cost summary (tokens, spend, models used) to stderr.

## 5. Budget enforcement (optional)

If you need hard spending limits that reject requests before they reach the provider, the [LLMKit proxy](https://github.com/smigolsmigol/llmkit) adds budget enforcement on top of cost tracking.

The proxy estimates cost before each request. If it would exceed the budget, the request is rejected before reaching the provider. Per-key or per-session scope.

```python
# route requests through the proxy for budget enforcement
client = anthropic.Anthropic(
    base_url="https://llmkit-proxy.smigolsmigol.workers.dev/v1",
    api_key="llmk_your_key_here",  # get one at llmkit-dashboard.vercel.app
)
```

The proxy supports all 11 providers (Anthropic, OpenAI, Gemini, Groq, Together, Fireworks, DeepSeek, Mistral, xAI, Ollama, OpenRouter) with automatic cost tracking, a dashboard, and anomaly detection alerts.

## Summary

| Approach | Setup | What you get |
|----------|-------|--------------|
| `tracked()` | `pip install llmkit-sdk` | Per-response cost estimates |
| `calculate_cost()` | Same | Standalone pricing lookup (730+ models) |
| JSONL parsing | Read `~/.claude/` files | Session/project cost totals |
| MCP server | `npx @f3d1/llmkit-mcp-server` | Real-time in-IDE cost visibility |
| Proxy | Free account | Budget enforcement + dashboard |